# ECG Encoder Pretraining on PTB-XL

**Purpose:** Pretrain only the ECG encoder (1D-ResNet + Transformer) on PTB-XL before plugging weights into the cardiac fusion model.

**Architecture trained:**
- `ResNet1D` → 1D-ResNet backbone (12 leads → temporal feature map)
- `ECGEncoder` → ResNet + AdaptivePool + SinusoidalPE + TransformerEncoder
- Thin classification head (discarded after pretraining — not saved to encoder checkpoint)

**What gets saved to Google Drive:**
- `ecg_encoder_pretrained.pt` — encoder weights only, ready to `load_state_dict` into fusion model
- `ecg_encoder_full_ckpt.pt` — full checkpoint with optimizer, epoch, config, history (for resuming)

**Dataset:** PTB-XL — 21,837 clinical 12-lead ECGs, resampled to 5000 samples

**Labels:** PTB-XL diagnostic superclass (5 classes): NORM, MI, STTC, CD, HYP

---
**Colab Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom
3. Weights auto-save to Google Drive after every best epoch

## Cell 0 — Mount Google Drive & Install Dependencies

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/ecg_encoder_pretrain'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

Mounted at /content/drive
Save directory: /content/drive/MyDrive/ecg_encoder_pretrain


In [3]:
!pip install wfdb --quiet

import torch
print(f'PyTorch   : {torch.__version__}')
print(f'CUDA avail: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')
    print(f'VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using     : {DEVICE}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 115.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
PyTorch   : 2.10

In [4]:
from google.colab import files
files.upload()  # upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"ibrahimhachemi","key":"0e8a667c57e588d00ea7f7ced1dab80a"}'}

In [5]:
!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
!kaggle datasets download -d rohitdwivedula/ptbxl-original-dataset

Dataset URL: https://www.kaggle.com/datasets/rohitdwivedula/ptbxl-original-dataset
License(s): unknown
100% 1.72G/1.72G [01:38<00:00, 18.7MB/s]



In [7]:
!unzip -q ptbxl-original-dataset.zip -d ptbxl

## Cell 1 — Configuration

In [10]:
import random
import numpy as np

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Data ───────────────────────────────────────────────────────────────────────
ECG_LEADS  = 12
ECG_LEN    = 5000        # PTB-XL is 1000 samples @ 100Hz -> resampled to 5000
BATCH_SIZE = 64

# PTB-XL superclass labels (5 classes)
PTBXL_CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
NUM_CLASSES    = len(PTBXL_CLASSES)

# ── Encoder Architecture ───────────────────────────────────────────────────────
# These MUST match your fusion model config exactly so weights transfer correctly
ECG_BASE_DIM      = 256
ECG_SEQ_LEN       = 128   # increased from 32 for better temporal resolution
ECG_TRANS_LAYERS  = 2     # full 2 transformer layers for pretraining
ATTN_HEADS        = 4
ATTN_DROPOUT      = 0.1

ECG_CFG = dict(
    transformer_d_model    = ECG_BASE_DIM,
    transformer_seq_len    = ECG_SEQ_LEN,
    transformer_nhead      = ATTN_HEADS,
    transformer_num_layers = ECG_TRANS_LAYERS,
    transformer_dim_ff     = ECG_BASE_DIM * 4,
    transformer_dropout    = ATTN_DROPOUT,
    resnet_in_channels     = ECG_LEADS,
    resnet_base_filters    = 32,
    resnet_blocks          = [2, 2, 2, 2],
    resnet_out_dim         = ECG_BASE_DIM,
)

# ── Training ───────────────────────────────────────────────────────────────────
EPOCHS       = 50
LR           = 3e-4
WEIGHT_DECAY = 1e-3
PATIENCE     = 10

# ── Paths ──────────────────────────────────────────────────────────────────────
PTBXL_DIR    = '/content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1'
ENCODER_CKPT = os.path.join(SAVE_DIR, 'ecg_encoder_pretrained.pt')
FULL_CKPT    = os.path.join(SAVE_DIR, 'ecg_encoder_full_ckpt.pt')

print('Config OK')
print(f'  ECG_SEQ_LEN      = {ECG_SEQ_LEN}')
print(f'  ECG_TRANS_LAYERS = {ECG_TRANS_LAYERS}')
print(f'  ECG_BASE_DIM     = {ECG_BASE_DIM}')
print(f'  NUM_CLASSES      = {NUM_CLASSES}  ->  {PTBXL_CLASSES}')

Config OK
  ECG_SEQ_LEN      = 128
  ECG_TRANS_LAYERS = 2
  ECG_BASE_DIM     = 256
  NUM_CLASSES      = 5  ->  ['NORM', 'MI', 'STTC', 'CD', 'HYP']


## Cell 2 — Download PTB-XL from PhysioNet

In [9]:
# ~1.8 GB download — takes 3-5 minutes on Colab
if not os.path.exists(PTBXL_DIR):
    print('Downloading PTB-XL dataset (~1.8 GB)...')
    os.makedirs('/content/ptb-xl', exist_ok=True)
    import subprocess
    result = subprocess.run([
        'wget', '-r', '-N', '-c', '-np', '--quiet',
        'https://physionet.org/files/ptb-xl/1.0.3/',
        '-P', '/content/'
    ], capture_output=True, text=True)
    if os.path.exists('/content/physionet.org/files/ptb-xl/1.0.3'):
        import shutil
        shutil.move('/content/physionet.org/files/ptb-xl/1.0.3', '/content/ptb-xl')
    print('Download complete.')
else:
    print(f'PTB-XL already at {PTBXL_DIR}')

print('Directory contents:')
for f in sorted(os.listdir(PTBXL_DIR))[:10]:
    print(f'  {f}')

KeyboardInterrupt: 

## Cell 3 — Load Metadata & Build 5-Class Labels

In [15]:
import pandas as pd
import ast

df = pd.read_csv("/content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/ptbxl_database.csv", index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(ast.literal_eval)

agg_df = pd.read_csv('/content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/scp_statements.csv', index_col=0)

def get_superclass(scp_dict):
    supers = {}
    for code, likelihood in scp_dict.items():
        if code in agg_df.index:
            sc = agg_df.loc[code, 'diagnostic_class']
            if isinstance(sc, str) and sc in PTBXL_CLASSES:
                supers[sc] = supers.get(sc, 0) + likelihood
    if not supers:
        return None
    return max(supers, key=supers.get)

df['superclass'] = df.scp_codes.apply(get_superclass)
df = df[df['superclass'].notna()].copy()
df['label'] = df['superclass'].map({c: i for i, c in enumerate(PTBXL_CLASSES)})

# PTB-XL official split: folds 1-8 = train, 9 = val, 10 = test
train_df = df[df.strat_fold <= 8].copy()
val_df   = df[df.strat_fold == 9].copy()
test_df  = df[df.strat_fold == 10].copy()

print(f'Total usable records : {len(df)}')
print(f'  Train : {len(train_df)}')
print(f'  Val   : {len(val_df)}')
print(f'  Test  : {len(test_df)}')
print(f'\nClass distribution (train):')
for c, n in train_df['superclass'].value_counts().items():
    print(f'  {c:6s}: {n:5d}')

Total usable records : 21430
  Train : 17111
  Val   : 2156
  Test  : 2163

Class distribution (train):
  NORM  :  7395
  MI    :  3262
  CD    :  2723
  STTC  :  2682
  HYP   :  1049


## Cell 4 — Dataset & DataLoaders

In [17]:
import wfdb
from scipy.signal import resample as scipy_resample
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


def load_ecg(row, ptbxl_dir, target_len=5000):
    """
    Load PTB-XL record -> (12, target_len) float32.
    Normalize BEFORE padding (fix from fusion model audit).
    """
    path = os.path.join(ptbxl_dir, row['filename_lr'])
    rec  = wfdb.rdrecord(path)
    sig  = rec.p_signal.astype(np.float32).T  # (12, T_orig)

    # Replace NaN per lead
    for i in range(sig.shape[0]):
        nan_mask = np.isnan(sig[i])
        if nan_mask.all():
            sig[i] = 0.0
        elif nan_mask.any():
            sig[i, nan_mask] = 0.0

    # ── NORMALIZE BEFORE RESAMPLING/PADDING ────────────────────────────
    mean  = sig.mean(axis=1, keepdims=True)
    std   = sig.std(axis=1,  keepdims=True)
    valid = std.squeeze() > 1e-6
    if valid.ndim == 0:
        valid = np.array([valid])
    sig[valid]  = np.clip((sig[valid]  - mean[valid])  / std[valid],  -8, 8)
    sig[~valid] = 0.0

    # ── Resample to target_len via scipy (high quality) ────────────────
    T_orig = sig.shape[1]
    if T_orig != target_len:
        sig = scipy_resample(sig, target_len, axis=1).astype(np.float32)

    # ── Pad / truncate (safety) ────────────────────────────────────────
    T = sig.shape[1]
    if T < target_len:
        sig = np.pad(sig, ((0, 0), (0, target_len - T)), mode='constant')
    elif T > target_len:
        sig = sig[:, :target_len]

    return sig  # (12, 5000) float32


def augment_ecg(ecg):
    """Training augmentation — same as fusion model."""
    ecg = ecg * np.random.uniform(0.85, 1.15)
    ecg = ecg + np.random.randn(*ecg.shape).astype(np.float32) * 0.04
    shift = np.random.randint(0, 300)
    ecg = np.concatenate([ecg[:, shift:],
                           np.zeros((ecg.shape[0], shift), dtype=np.float32)], axis=1)
    if np.random.rand() > 0.6:
        n_drop   = np.random.randint(1, 3)
        drop_idx = np.random.choice(12, n_drop, replace=False)
        ecg[drop_idx] = 0.0
    if np.random.rand() > 0.5:
        ecg = -ecg
    return np.clip(ecg, -8.0, 8.0)


class PTBXLDataset(Dataset):
    def __init__(self, df, ptbxl_dir, augment=False):
        self.df        = df.reset_index()
        self.ptbxl_dir = ptbxl_dir
        self.augment   = augment
        self.labels    = df['label'].values.astype(np.int64)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ecg = load_ecg(row, self.ptbxl_dir, target_len=ECG_LEN)
        ecg = np.nan_to_num(ecg, nan=0.0, posinf=8.0, neginf=-8.0)
        ecg = np.clip(ecg, -8.0, 8.0)
        if self.augment:
            ecg = augment_ecg(ecg)
        return (torch.from_numpy(ecg),
                torch.tensor(int(row['label']), dtype=torch.long))


train_ds = PTBXLDataset(train_df, PTBXL_DIR, augment=True)
val_ds   = PTBXLDataset(val_df,   PTBXL_DIR, augment=False)
test_ds  = PTBXLDataset(test_df,  PTBXL_DIR, augment=False)

# Sampler for class balance
counts_c = np.bincount(train_ds.labels, minlength=NUM_CLASSES).astype(float)
counts_c = np.where(counts_c == 0, 1.0, counts_c)
sample_weights = (1.0 / counts_c)[train_ds.labels]
sampler = WeightedRandomSampler(sample_weights, len(train_ds), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train : {len(train_ds):>6d} samples  |  {len(train_loader)} batches')
print(f'Val   : {len(val_ds):>6d} samples  |  {len(val_loader)} batches')
print(f'Test  : {len(test_ds):>6d} samples  |  {len(test_loader)} batches')

Train :  17111 samples  |  268 batches
Val   :   2156 samples  |  34 batches
Test  :   2163 samples  |  34 batches


## Cell 5 — ECG Encoder Architecture (exact copy from fusion model)

In [18]:
import math
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import List

# ─────────────────────────────────────────────────────────────────────────────
# Building blocks — IDENTICAL to fusion_model_v4_5class_staged.ipynb
# DO NOT MODIFY — any change here must be mirrored in the fusion model
# ─────────────────────────────────────────────────────────────────────────────

class Conv1dBnRelu(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size,
                      stride=stride, padding=padding, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, 3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))


class ResNet1D(nn.Module):
    def __init__(self, in_channels, base_filters, block_counts, out_dim):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_filters, kernel_size=7,
                      stride=2, padding=3, bias=False),
            nn.BatchNorm1d(base_filters),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        stages, ch_in, ch_out = [], base_filters, base_filters
        for stage_idx, n_blocks in enumerate(block_counts):
            stride = 2 if stage_idx > 0 else 1
            stages.append(self._make_stage(ch_in, ch_out, n_blocks, stride))
            ch_in  = ch_out
            ch_out = ch_out * 2
        self.stages = nn.Sequential(*stages)
        final_ch = base_filters * (2 ** (len(block_counts) - 1))
        self.channel_adapter = nn.Sequential(
            nn.Conv1d(final_ch, out_dim, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
        )

    @staticmethod
    def _make_stage(in_channels, out_channels, n_blocks, stride):
        blocks = [ResidualBlock1D(in_channels, out_channels, stride=stride)]
        for _ in range(1, n_blocks):
            blocks.append(ResidualBlock1D(out_channels, out_channels))
        return nn.Sequential(*blocks)

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        return self.channel_adapter(x)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float()
                             * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class ECGEncoder(nn.Module):
    """
    Exact copy from fusion_model_v4_5class_staged.ipynb.
    (B, 12, T) -> ResNet1D -> AdaptivePool -> SinPE -> Transformer -> (B, seq_len, d_model)
    """
    def __init__(self, cfg: dict):
        super().__init__()
        d_model  = cfg['transformer_d_model']
        seq_len  = cfg['transformer_seq_len']
        n_head   = cfg['transformer_nhead']
        n_layers = cfg['transformer_num_layers']
        dim_ff   = cfg['transformer_dim_ff']
        dropout  = cfg['transformer_dropout']

        self.backbone = ResNet1D(
            in_channels  = cfg['resnet_in_channels'],
            base_filters = cfg['resnet_base_filters'],
            block_counts = cfg['resnet_blocks'],
            out_dim      = cfg['resnet_out_dim'],
        )
        self.adaptive_pool = nn.AdaptiveAvgPool1d(seq_len)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len=seq_len, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_head, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers, norm=nn.LayerNorm(d_model)
        )

    def forward(self, x: Tensor) -> Tensor:
        features = self.backbone(x)
        features = self.adaptive_pool(features)
        tokens   = features.permute(0, 2, 1).contiguous()
        tokens   = self.pos_enc(tokens)
        tokens   = self.transformer(tokens)
        return tokens  # (B, seq_len, d_model)


# ─────────────────────────────────────────────────────────────────────────────
# Classification wrapper — head is ONLY used during pretraining, NOT saved
# ─────────────────────────────────────────────────────────────────────────────
class ECGClassifier(nn.Module):
    def __init__(self, cfg, num_classes):
        super().__init__()
        self.encoder = ECGEncoder(cfg)
        d_model = cfg['transformer_d_model']
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, ecg):
        tokens = self.encoder(ecg)     # (B, seq_len, d_model)
        pooled = tokens.mean(dim=1)    # (B, d_model)
        return self.head(pooled)       # (B, num_classes)

    def count_params(self):
        enc_p  = sum(p.numel() for p in self.encoder.parameters())
        head_p = sum(p.numel() for p in self.head.parameters())
        print(f'Encoder params : {enc_p/1e6:.3f} M  [will be saved]')
        print(f'Head params    : {head_p/1e3:.1f} K  [discarded after pretraining]')
        print(f'Total          : {(enc_p + head_p)/1e6:.3f} M')


# Instantiate and dry-run
model = ECGClassifier(ECG_CFG, NUM_CLASSES).to(DEVICE)
model.count_params()

with torch.no_grad():
    dummy = torch.randn(2, 12, ECG_LEN).to(DEVICE)
    out   = model(dummy)
    print(f'\nDry-run: input={list(dummy.shape)}  output={list(out.shape)}  (expected [2, {NUM_CLASSES}])')

/tmp/ipykernel_9894/2005832541.py:127: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Encoder params : 2.612 M  [will be saved]
Head params    : 34.1 K  [discarded after pretraining]
Total          : 2.646 M

Dry-run: input=[2, 12, 5000]  output=[2, 5]  (expected [2, 5])


## Cell 6 — Loss, Optimizer, Scheduler

In [19]:
from sklearn.metrics import f1_score

# ── Class-weighted loss ────────────────────────────────────────────────────────
counts_train = np.bincount(train_ds.labels, minlength=NUM_CLASSES).astype(float)
counts_train = np.where(counts_train == 0, 1.0, counts_train)
class_weights = counts_train.sum() / (NUM_CLASSES * counts_train)
class_weights = np.clip(class_weights, 1.0, 5.0)
w_tensor  = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=w_tensor, label_smoothing=0.05)

print('Class weights:')
for c, w in zip(PTBXL_CLASSES, class_weights):
    print(f'  {c:6s}: {w:.3f}')

# ── AdamW with no-decay group for bias / LayerNorm / BatchNorm ───────────────
no_decay = {'bias', 'LayerNorm.weight', 'BatchNorm1d.weight',
            'BatchNorm1d.bias', 'LayerNorm.bias'}
decay_params    = [p for n, p in model.named_parameters()
                   if p.requires_grad and not any(nd in n for nd in no_decay)]
no_decay_params = [p for n, p in model.named_parameters()
                   if p.requires_grad and any(nd in n for nd in no_decay)]

optimizer = torch.optim.AdamW(
    [
        {'params': decay_params,    'weight_decay': WEIGHT_DECAY},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ],
    lr=LR,
)

# ── 3-epoch linear warmup then cosine annealing ───────────────────────────────
warmup_epochs    = 3
cosine_epochs    = EPOCHS - warmup_epochs
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cosine_epochs, eta_min=1e-6
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
)

scaler = torch.amp.GradScaler() if DEVICE.type == 'cuda' else None

print(f'\nOptimizer  : AdamW  lr={LR}  wd={WEIGHT_DECAY} (bias/LN excluded)')
print(f'Scheduler  : {warmup_epochs}-epoch warmup -> CosineAnnealingLR ({cosine_epochs} epochs)')
print(f'Loss       : CrossEntropyLoss(weight=..., label_smoothing=0.05)')
print(f'AMP scaler : {scaler is not None}')

Class weights:
  NORM  : 1.000
  MI    : 1.049
  STTC  : 1.276
  CD    : 1.257
  HYP   : 3.262

Optimizer  : AdamW  lr=0.0003  wd=0.001 (bias/LN excluded)
Scheduler  : 3-epoch warmup -> CosineAnnealingLR (47 epochs)
Loss       : CrossEntropyLoss(weight=..., label_smoothing=0.05)
AMP scaler : True


## Cell 7 — Training & Evaluation Functions

In [21]:
import time


def run_epoch(model, loader, optimizer=None, scaler=None):
    """One train or eval pass. Returns (avg_loss, accuracy)."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = correct = total = 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for ecg, labels in loader:
            ecg, labels = ecg.to(DEVICE), labels.to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            if DEVICE.type == 'cuda':
                with torch.autocast('cuda', torch.float16):
                    logits = model(ecg)
                    loss   = criterion(logits, labels)
            else:
                logits = model(ecg)
                loss   = criterion(logits, labels)

            if torch.isnan(loss) or torch.isinf(loss):
                if is_train:
                    optimizer.zero_grad()
                continue

            if is_train:
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += labels.size(0)

    if total == 0:
        return float('nan'), 0.0
    return total_loss / total, correct / total


def evaluate_full(model, loader):
    """Returns (true_labels np.array, pred_probs np.array)."""
    model.eval()
    all_probs, all_true = [], []
    with torch.no_grad():
        for ecg, labels in loader:
            ecg    = ecg.to(DEVICE)
            logits = model(ecg)
            probs  = torch.softmax(logits.float(), dim=1).cpu().numpy()
            all_probs.append(probs)
            all_true.extend(labels.tolist())
    return np.array(all_true), np.vstack(all_probs)


print('Training functions defined.')

Training functions defined.


## Cell 8 — Train

In [ ]:
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'val_f1':     [],
}

best_val_f1 = 0.0
no_improve  = 0

print('=' * 70)
print(f'  ECG Encoder Pretraining on PTB-XL  ({NUM_CLASSES} classes)')
print(f'  Epochs={EPOCHS}  LR={LR}  Batch={BATCH_SIZE}  Patience={PATIENCE}')
print('=' * 70)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = run_epoch(model, train_loader, optimizer, scaler)
    val_loss,   val_acc   = run_epoch(model, val_loader)
    scheduler.step()

    val_true, val_probs = evaluate_full(model, val_loader)
    val_preds = val_probs.argmax(-1)
    val_f1    = f1_score(val_true, val_preds, average='macro', zero_division=0)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    cur_lr  = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0

    flag = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        no_improve  = 0

        # ── Save encoder weights ONLY (no head) ──────────────────────────
        torch.save(model.encoder.state_dict(), ENCODER_CKPT)

        # ── Save full resumable checkpoint ────────────────────────────────
        torch.save({
            'epoch'             : epoch,
            'encoder_state_dict': model.encoder.state_dict(),
            'full_state_dict'   : model.state_dict(),
            'optimizer_state'   : optimizer.state_dict(),
            'scheduler_state'   : scheduler.state_dict(),
            'best_val_f1'       : best_val_f1,
            'val_loss'          : val_loss,
            'history'           : history,
            'ecg_cfg'           : ECG_CFG,
            'ptbxl_classes'     : PTBXL_CLASSES,
            'ecg_len'           : ECG_LEN,
        }, FULL_CKPT)

        flag = '  <- BEST  [saved to Drive]'
    else:
        no_improve += 1

    print(f'  {epoch:03d}/{EPOCHS} | '
          f'tr {train_loss:.3f} acc={train_acc:.3f} | '
          f'val {val_loss:.3f} acc={val_acc:.3f} f1={val_f1:.3f} | '
          f'lr={cur_lr:.2e} | {elapsed:.1f}s{flag}')

    if no_improve >= PATIENCE:
        print(f'\n  Early stopping at epoch {epoch}  (patience={PATIENCE} exhausted)')
        break

print(f'\n  Pretraining complete')
print(f'  Best val Macro-F1       : {best_val_f1:.4f}')
print(f'  Encoder weights saved   : {ENCODER_CKPT}')
print(f'  Full checkpoint saved   : {FULL_CKPT}')

  ECG Encoder Pretraining on PTB-XL  (5 classes)
  Epochs=50  LR=0.0003  Batch=64  Patience=10
  001/50 | tr 1.353 acc=0.308 | val 1.364 acc=0.393 f1=0.359 | lr=1.20e-04 | 259.3s  <- BEST  [saved to Drive]
  002/50 | tr 1.167 acc=0.469 | val 1.334 acc=0.435 f1=0.420 | lr=2.10e-04 | 247.7s  <- BEST  [saved to Drive]


## Cell 9 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Cross-Entropy Loss')
axes[0].legend()
axes[0].set_xlabel('Epoch')

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'],   label='Val')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].set_xlabel('Epoch')

axes[2].plot(history['val_f1'], label='Val Macro-F1', color='green')
axes[2].axhline(best_val_f1, linestyle='--', color='red',
                alpha=0.6, label=f'Best = {best_val_f1:.3f}')
axes[2].set_title('Val Macro-F1')
axes[2].legend()
axes[2].set_xlabel('Epoch')

plt.suptitle('ECG Encoder Pretraining — PTB-XL', fontsize=13, fontweight='bold')
plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, 'pretrain_history.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved: {plot_path}')

## Cell 10 — Test Set Evaluation

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score)
from sklearn.preprocessing import label_binarize
import seaborn as sns

# Load best encoder into model
model.encoder.load_state_dict(torch.load(ENCODER_CKPT, map_location=DEVICE))
print(f'Loaded best encoder from {ENCODER_CKPT}')

test_true, test_probs = evaluate_full(model, test_loader)
test_preds  = test_probs.argmax(-1)
test_f1     = f1_score(test_true, test_preds, average='macro', zero_division=0)

test_true_bin = label_binarize(test_true, classes=list(range(NUM_CLASSES)))
test_auroc    = roc_auc_score(test_true_bin, test_probs,
                              average='macro', multi_class='ovr')

print('\n' + '=' * 60)
print('  TEST SET RESULTS — ECG Encoder  (PTB-XL, 5-class)')
print('=' * 60)
print(f'  Macro-F1 : {test_f1:.4f}')
print(f'  AUROC    : {test_auroc:.4f}')
print()
print(classification_report(test_true, test_preds,
                             target_names=PTBXL_CLASSES, digits=3))

cm = confusion_matrix(test_true, test_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=PTBXL_CLASSES,
            yticklabels=PTBXL_CLASSES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('ECG Encoder — Test Confusion Matrix', fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(SAVE_DIR, 'pretrain_confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Confusion matrix saved: {cm_path}')

## Cell 11 — How to Load Pretrained Weights in Fusion Model

In [ ]:
# Verify the saved encoder file loads cleanly
saved = torch.load(ENCODER_CKPT, map_location='cpu')
print(f'Encoder checkpoint: {ENCODER_CKPT}')
print(f'  Keys in state_dict : {len(saved)}')
print(f'  First 5 keys:')
for k in list(saved.keys())[:5]:
    print(f'    {k:55s}  {list(saved[k].shape)}')

# Verify clean load
test_enc = ECGEncoder(ECG_CFG)
missing, unexpected = test_enc.load_state_dict(saved, strict=True)
print(f'\nClean load into fresh ECGEncoder: OK')
print(f'  Missing keys   : {len(missing)}')
print(f'  Unexpected keys: {len(unexpected)}')

print(f"""
{'='*65}
  PASTE THIS INTO fusion_model_v4_5class_staged.ipynb
  In the model instantiation cell, AFTER model = CardiacFusionModel().to(DEVICE)
{'='*65}

PRETRAINED_ECG_CKPT = r'{ENCODER_CKPT}'

state = torch.load(PRETRAINED_ECG_CKPT, map_location=DEVICE)
missing, unexpected = model.ecg_encoder.load_state_dict(state, strict=True)
print(f'ECG encoder loaded from PTB-XL pretrain')
print(f'  Missing={{len(missing)}}  Unexpected={{len(unexpected)}}')

# Stage 1 (frozen encoders) will now start from pretrained features.
# Stage 2 (ECG encoder unfrozen) fine-tunes from PTB-XL weights, not random init.
# This is the key benefit — the encoder already knows ECG morphology.
{'='*65}
""")

## Cell 12 — (Optional) Resume Training After Colab Disconnect

In [ ]:
# Set RESUME = True only if Colab disconnected mid-training.
# Requires re-running Cells 0-7 first to rebuild model and optimizer.

RESUME = False

if RESUME and os.path.exists(FULL_CKPT):
    ckpt = torch.load(FULL_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt['full_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    best_val_f1 = ckpt['best_val_f1']
    history     = ckpt['history']
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed from epoch {ckpt["epoch"]}')
    print(f'  best_val_f1 = {best_val_f1:.4f}')
    print(f'  Will continue from epoch {start_epoch}')
    print(f'  Re-run Cell 8 loop starting at start_epoch')
else:
    print('RESUME=False — skipping.')